# Two-Part Model Baseline — Cartier QTEM Data Challenge

**Architettura Hurdle Model:**
- **Parte 1** — Logistic Regression: classifica P(TARGET_3Y > 0)
- **Parte 2** — XGBoost Regressor: stima E[log(TARGET_3Y) | TARGET_3Y > 0]
- **Predizione finale**: P(spende) × E[quanto spende | spende]

| | Valore | Baseline | Lift |
|---|---|---|---|
| PR-AUC (classificatore) | 0.2691 | 0.0483 | **5.6x** |
| ROC-AUC | 0.8509 | 0.5000 | — |
| Recall top decile | 50.2% | — | — |
| RMSE log-space | 1.1346 | 1.2900 | **12.0%** |
| MAE EUR (positivi) | 5.178 | — | — |
| Top 10% → revenue catturata | 60.2% | — | — |

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

ROOT       = os.path.abspath('..')
FEAT_DIR   = os.path.join(ROOT, 'data', 'features')
MODELS_DIR = os.path.join(ROOT, 'output', 'models')
TABLES_DIR = os.path.join(ROOT, 'output', 'tables')
PLOTS_DIR  = os.path.join(ROOT, 'output', 'plots')

# Carica risultati pre-calcolati
clf_results = pd.read_csv(os.path.join(TABLES_DIR, 'classifier_results.csv'))
reg_results = pd.read_csv(os.path.join(TABLES_DIR, 'regressor_results.csv'))
preds       = pd.read_csv(os.path.join(TABLES_DIR, 'test_predictions.csv'))
summary     = pd.read_csv(os.path.join(TABLES_DIR, 'model_summary.csv'))
clf_imp     = pd.read_csv(os.path.join(TABLES_DIR, 'classifier_feature_importance.csv'))
reg_imp     = pd.read_csv(os.path.join(TABLES_DIR, 'regressor_feature_importance.csv'))
rev_capture = pd.read_csv(os.path.join(TABLES_DIR, 'revenue_capture.csv'))

print('Dati caricati correttamente')
print(f'Predizioni test: {preds.shape}')

## Parte 1 — Classificatore: Precision-Recall Curve

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

y_true = preds['BINARY_TARGET_actual'].values
y_prob = preds['P_SPEND'].values

precision, recall, _ = precision_recall_curve(y_true, y_prob)
pr_auc    = average_precision_score(y_true, y_prob)
baseline  = y_true.mean()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recall, precision, color='steelblue', lw=2,
        label=f'Logistic Regression (PR-AUC = {pr_auc:.4f})')
ax.axhline(baseline, color='gray', linestyle='--',
           label=f'Baseline casuale (prevalenza = {baseline:.4f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — Classificatore P(TARGET_3Y > 0)')
ax.legend()
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'pr_curve.png'), dpi=120, bbox_inches='tight')
plt.show()
print(f'PR-AUC: {pr_auc:.4f}  |  Lift: {pr_auc/baseline:.1f}x sul baseline')

## Parte 1 — Feature importance (coefficienti LR)

In [ ]:
top15_lr = clf_imp.head(15).sort_values('abs_coefficient')

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['tomato' if c < 0 else 'steelblue' for c in top15_lr['coefficient']]
ax.barh(top15_lr['feature'], top15_lr['coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 15 feature — Coefficienti Logistic Regression')
ax.set_xlabel('Coefficiente (positivo = aumenta P(spende))')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'lr_coefficients.png'), dpi=120, bbox_inches='tight')
plt.show()

## Parte 2 — Feature importance XGBoost

In [ ]:
top15_xgb = reg_imp.head(15).sort_values('importance')

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top15_xgb['feature'], top15_xgb['importance'], color='darkorange')
ax.set_title('Top 15 feature — XGBoost Gain Importance (Regressore)')
ax.set_xlabel('Gain importance')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'xgb_importance.png'), dpi=120, bbox_inches='tight')
plt.show()

## Revenue Capture Curve

Percentuale di revenue totale catturata ordinando i clienti per score predetto (Combined Prediction decrescente).

In [ ]:
preds_sorted    = preds.sort_values('COMBINED_PREDICTION', ascending=False).reset_index(drop=True)
total_revenue   = preds['TARGET_3Y_actual'].sum()
cum_revenue     = preds_sorted['TARGET_3Y_actual'].cumsum() / total_revenue
cum_population  = (preds_sorted.index + 1) / len(preds_sorted)

# Curva lorenz casuale (baseline)
random_curve = cum_population

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(cum_population * 100, cum_revenue * 100,
        color='steelblue', lw=2, label='Two-Part Model')
ax.plot([0, 100], [0, 100], 'gray', linestyle='--', label='Baseline casuale')

# Annota punti chiave
for pct in [0.01, 0.05, 0.10, 0.20]:
    idx = int(len(preds_sorted) * pct) - 1
    rev = cum_revenue.iloc[idx] * 100
    ax.annotate(f'Top {pct:.0%}: {rev:.0f}% rev.',
                xy=(pct * 100, rev),
                xytext=(pct * 100 + 3, rev - 5),
                fontsize=8, color='steelblue',
                arrowprops=dict(arrowstyle='->', color='steelblue', lw=1))

ax.set_xlabel('% clienti (ordinati per score)')
ax.set_ylabel('% revenue catturata')
ax.set_title('Revenue Capture Curve — Two-Part Model')
ax.legend()
ax.set_xlim([0, 100])
ax.set_ylim([0, 100])
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'revenue_capture.png'), dpi=120, bbox_inches='tight')
plt.show()

print('Revenue capture table:')
print(rev_capture.to_string(index=False))

## Predizioni vs Attuale — scatter sui positivi

In [ ]:
pos = preds[preds['BINARY_TARGET_actual'] == 1].copy()
log_actual = np.log1p(pos['TARGET_3Y_actual'])
log_pred   = np.log1p(pos['COMBINED_PREDICTION'].clip(lower=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Log-scale scatter
sample = pos.sample(min(5000, len(pos)), random_state=42)
axes[0].scatter(np.log1p(sample['TARGET_3Y_actual']),
                np.log1p(sample['COMBINED_PREDICTION'].clip(lower=0)),
                alpha=0.3, s=5, color='steelblue')
max_val = max(log_actual.max(), log_pred.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', lw=1, label='Perfetto')
axes[0].set_xlabel('log1p(TARGET_3Y actual)')
axes[0].set_ylabel('log1p(Combined Prediction)')
axes[0].set_title('Predetto vs Attuale (log-scale, campione 5k)')
axes[0].legend()

# Distribuzione errori EUR
err_eur = (pos['COMBINED_PREDICTION'].clip(lower=0) - pos['TARGET_3Y_actual'])
err_clip = err_eur.clip(-20000, 20000)
axes[1].hist(err_clip, bins=60, color='steelblue', alpha=0.7, edgecolor='none')
axes[1].axvline(0, color='red', linestyle='--', lw=1)
axes[1].set_xlabel('Errore EUR (clipped +-20k)')
axes[1].set_ylabel('Frequenza')
axes[1].set_title(f'Distribuzione errori (MAE={pos["TARGET_3Y_actual"].sub(pos["COMBINED_PREDICTION"].clip(lower=0)).abs().median():,.0f} EUR median)')

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'prediction_vs_actual.png'), dpi=120, bbox_inches='tight')
plt.show()

## Summary — Mid-Term Presentation

In [ ]:
print('=== TWO-PART MODEL BASELINE — RISULTATI FINALI ===')
print()
for _, row in summary.iterrows():
    print(f"  [{row['section'][:30]:30s}]  {row['metric']:32s}  {row['value']}")

print()
print('PROSSIMI PASSI (post mid-term):')
print('  1. Sostituire LR con XGBoost classifier (evita scalatura, gestisce non-linearita)')
print('  2. Usare validation fold interna per early stopping (non test set)')
print('  3. Tuning iperparametri su entrambe le parti')
print('  4. Parsing ALL_PURCHASED_* per feature sequenziali')
print('  5. Estendere a TARGET_5Y per CLV a lungo termine')